# EDA: NBA 3.1 HCP Feature Engineering Results

**Purpose:** Run exploratory data analysis on the output of notebooks 26 (NBRx), 27 (NRx), 28 (TRx), and 31 (Competitor Share) to validate feature generation quality.

**Analyses:**
1. Row/column counts per feature table
2. Null percentage per feature
3. Sparsity (zero percentage) per feature
4. Distribution stats (min, max, mean, stddev, median)
5. Date range coverage
6. Cross-notebook feature summary
7. Save EDA summary report

**Runs after:** All 4 feature notebooks complete on full HCP universe

In [0]:
# Cell 1: Configuration and load all 4 output tables
from pyspark.sql import functions as F
from pyspark.sql.functions import col, count, when, min as smin, max as smax, avg, stddev, lit
import pandas as pd

VOLUME_BASE = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume"
FEATURE_PATHS = {
    "NBRx": f"{VOLUME_BASE}/notebook_output/hcp_features/nbrx_features",
    "NRx": f"{VOLUME_BASE}/notebook_output/hcp_features/nrx_features",
    "TRx": f"{VOLUME_BASE}/notebook_output/hcp_features/trx_features",
    "Competitor_Share": f"{VOLUME_BASE}/notebook_output/hcp_features/competitor_share_features"
}
EDA_OUTPUT = f"{VOLUME_BASE}/notebook_output/hcp_features/eda_summary"

# Load all tables
feature_tables = {}
for name, path in FEATURE_PATHS.items():
    try:
        feature_tables[name] = spark.read.format("delta").load(path)
        print(f"{name}: loaded from {path}")
    except Exception as e:
        print(f"{name}: FAILED to load - {e}")

NON_FEATURE_COLS = {"PROVIDER_ID", "SVC_DT_M", "PULL_DT", "HCP_COHORT_ID"}
print(f"Loaded {len(feature_tables)} feature tables")

In [0]:
# Cell 2: Basic stats per table (row count, column count, date range, HCP count)
summary_rows = []
for name, df in feature_tables.items():
    total_cols = len(df.columns)
    feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
    row_count = df.count()
    hcp_count = df.select("PROVIDER_ID").distinct().count()
    date_range = df.agg(smin("SVC_DT_M").alias("min_date"), smax("SVC_DT_M").alias("max_date")).collect()[0]
    summary_rows.append({
        "Table": name, "Rows": row_count, "Feature_Cols": len(feature_cols),
        "Total_Cols": total_cols, "Distinct_HCPs": hcp_count,
        "Min_Date": str(date_range["min_date"]), "Max_Date": str(date_range["max_date"])
    })
    print(f"{name}: {row_count:,} rows, {len(feature_cols)} features, {hcp_count:,} HCPs, {date_range['min_date']} to {date_range['max_date']}")

df_summary = pd.DataFrame(summary_rows)
print("\nSummary:")
print(df_summary.to_string(index=False))

In [0]:
# Cell 3: Null percentage and sparsity (zero percentage) per feature
# Sample 100K rows for efficiency on large tables
SAMPLE_SIZE = 100000

all_null_sparsity = []
for name, df in feature_tables.items():
    feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
    # Sample for efficiency
    if df.count() > SAMPLE_SIZE:
        df_sample = df.sample(withReplacement=False, fraction=SAMPLE_SIZE / df.count(), seed=42)
    else:
        df_sample = df
    sample_count = df_sample.count()
    
    # Build null and zero count expressions
    null_exprs = [count(when(col(c).isNull(), c)).alias(c) for c in feature_cols]
    zero_exprs = [count(when(col(c) == 0, c)).alias(c) for c in feature_cols]
    
    null_counts = df_sample.agg(*null_exprs).collect()[0].asDict()
    zero_counts = df_sample.agg(*zero_exprs).collect()[0].asDict()
    
    for c in feature_cols:
        null_pct = (null_counts.get(c, 0) / sample_count) * 100
        zero_pct = (zero_counts.get(c, 0) / sample_count) * 100
        all_null_sparsity.append({
            "Table": name, "Feature": c,
            "Null_Pct": round(null_pct, 2), "Zero_Pct": round(zero_pct, 2),
            "Sparsity_Score": round(null_pct + zero_pct, 2)
        })
    print(f"{name}: computed null/sparsity for {len(feature_cols)} features")

df_null_sparsity = pd.DataFrame(all_null_sparsity)
print(f"\nTotal features analyzed: {len(df_null_sparsity)}")
print("\nTop 20 most sparse features:")
print(df_null_sparsity.nlargest(20, 'Sparsity_Score')[['Table', 'Feature', 'Null_Pct', 'Zero_Pct']].to_string(index=False))

In [0]:
# Cell 4: Distribution stats (min, max, mean, stddev) per feature
all_dist_stats = []
for name, df in feature_tables.items():
    feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
    # Use same sample as above for efficiency
    if df.count() > SAMPLE_SIZE:
        df_sample = df.sample(withReplacement=False, fraction=SAMPLE_SIZE / df.count(), seed=42)
    else:
        df_sample = df
    
    # Build stats expressions (only for numeric columns)
    stat_exprs = []
    for c in feature_cols:
        stat_exprs.extend([
            smin(col(c).cast("double")).alias(f"{c}__min"),
            smax(col(c).cast("double")).alias(f"{c}__max"),
            avg(col(c).cast("double")).alias(f"{c}__mean"),
            stddev(col(c).cast("double")).alias(f"{c}__stddev")
        ])
    
    stats = df_sample.agg(*stat_exprs).collect()[0].asDict()
    
    for c in feature_cols:
        all_dist_stats.append({
            "Table": name, "Feature": c,
            "Min": stats.get(f"{c}__min"),
            "Max": stats.get(f"{c}__max"),
            "Mean": round(stats.get(f"{c}__mean") or 0, 4),
            "StdDev": round(stats.get(f"{c}__stddev") or 0, 4)
        })
    print(f"{name}: computed distribution stats for {len(feature_cols)} features")

df_dist_stats = pd.DataFrame(all_dist_stats)
print(f"\nTotal features analyzed: {len(df_dist_stats)}")
print("\nFeatures with highest mean values:")
print(df_dist_stats.nlargest(20, 'Mean')[['Table', 'Feature', 'Min', 'Max', 'Mean', 'StdDev']].to_string(index=False))

In [0]:
# Cell 5: Feature family breakdown (count features by prefix/dimension)
feature_families = []
for name, df in feature_tables.items():
    feature_cols = [c for c in df.columns if c not in NON_FEATURE_COLS]
    # Group by dimension prefix
    dim_counts = {}
    for c in feature_cols:
        if "PRODUCT_GROUP" in c or any(kw in c for kw in ["SGLT2I", "MRA", "HF_DRUG"]):
            dim = "PRODUCT_GROUP/Competitor"
        elif "SPECIALTY_GROUP" in c:
            dim = "SPECIALTY_GROUP"
        elif "SWITCH_FROM" in c:
            dim = "SWITCH_FROM"
        elif "ADD_ON_TO" in c:
            dim = "ADD_ON_TO"
        elif "AGE_GROUP" in c:
            dim = "AGE_GROUP"
        elif "PAY_TYPE" in c:
            dim = "PAY_TYPE"
        elif "SOB" in c:
            dim = "SOURCE_OF_BUSINESS"
        elif "CHANNEL" in c:
            dim = "RX_CHANNEL"
        elif "INDICATION_GROUP" in c:
            dim = "INDICATION_GROUP"
        elif "HYPERTENSION_FLAG" in c:
            dim = "HYPERTENSION_FLAG"
        elif "GENDER" in c:
            dim = "GENDER"
        else:
            dim = "Other"
        dim_counts[dim] = dim_counts.get(dim, 0) + 1
    
    for dim, cnt in sorted(dim_counts.items(), key=lambda x: -x[1]):
        feature_families.append({"Table": name, "Dimension": dim, "Feature_Count": cnt})

df_families = pd.DataFrame(feature_families)
print("Feature family breakdown:")
print(df_families.to_string(index=False))
print(f"\nTotal features across all tables: {df_families['Feature_Count'].sum()}")

In [0]:
# Cell 6: Save EDA summary report
# Combine all EDA results into a summary Delta table
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, IntegerType

# Summary table
df_summary_spark = spark.createDataFrame(df_summary)
df_summary_spark.write.mode("overwrite").format("delta").save(f"{EDA_OUTPUT}/table_summary")
print("Saved table_summary")

# Null/sparsity table
df_null_sparsity_spark = spark.createDataFrame(df_null_sparsity)
df_null_sparsity_spark.write.mode("overwrite").format("delta").save(f"{EDA_OUTPUT}/null_sparsity")
print("Saved null_sparsity")

# Distribution stats
df_dist_stats_spark = spark.createDataFrame(df_dist_stats)
df_dist_stats_spark.write.mode("overwrite").format("delta").save(f"{EDA_OUTPUT}/distribution_stats")
print("Saved distribution_stats")

# Feature families
df_families_spark = spark.createDataFrame(df_families)
df_families_spark.write.mode("overwrite").format("delta").save(f"{EDA_OUTPUT}/feature_families")
print("Saved feature_families")

print(f"\nEDA complete. All reports saved to: {EDA_OUTPUT}")
print(f"Total features analyzed: {len(df_null_sparsity)} across {len(feature_tables)} tables")